## NLP for Supervised Learning
Sentimentanalyse von Kaffeebecher-Bewertungen zur Klassifizierung in positive und negative Reviews.

### Read in cappuccino cup review data

In [1]:
import nltk
import pandas as pd

In [3]:
data = pd.read_csv('coffee.csv')
data.head()

,user_id,stars,reviews
0,A2XP9IN4JOMROD,1,I wanted to love this. I was even prepared for...
1,A2TS09JCXNV1VD,5,Grove Square Cappuccino Cups were excellent. T...
2,AJ3L5J7GN09SV,2,I bought the Grove Square hazelnut cappuccino ...
3,A3CZD34ZTUJME7,1,"I love my Keurig, and I love most of the Keuri..."
4,AWKN396SHAQGP,1,It's a powdered drink. No filter in k-cup.<br ...


### Exploratory data analysis

In [7]:
data.stars.value_counts(normalize=True)

stars
5    0.568266
1    0.177122
4    0.119926
2    0.083026
3    0.051661
Name: proportion, dtype: float64

In [9]:
import numpy as np

# 3 Sterne löchen
data = data[data.stars!=3]

# Setze 4- und 5-Sterne-Bewertungen auf positiv, die übrigen auf negativ.
data['sentiment'] = np.where(data['stars'] >= 4, 'positive', 'negative')

# Nur die Spalten „Sentiment“ und „Bewertungen“ einbeziehen.
data = data[['sentiment', 'reviews']]
data.head()

,sentiment,reviews
0,negative,I wanted to love this. I was even prepared for...
1,positive,Grove Square Cappuccino Cups were excellent. T...
2,negative,I bought the Grove Square hazelnut cappuccino ...
3,negative,"I love my Keurig, and I love most of the Keuri..."
4,negative,It's a powdered drink. No filter in k-cup.<br ...


In [13]:
data.sentiment.value_counts(normalize=True)

sentiment
positive    0.725681
negative    0.274319
Name: proportion, dtype: float64

### Preprocess the text

In [17]:
# Textvorverarbeitungsschritte – Entfernen von Zahlen, Großbuchstaben und Satzzeichen.

import re
import string

alphanumeric = lambda x: re.sub('\w*\d\w*', ' ', x)
punc_lower = lambda x: re.sub('[%s]' % re.escape(string.punctuation), ' ', x.lower())

data['reviews'] = data.reviews.map(alphanumeric).map(punc_lower)
data.head()

<>:5: SyntaxWarning: invalid escape sequence '\w'
<>:5: SyntaxWarning: invalid escape sequence '\w'
C:\Users\HP\AppData\Local\Temp\ipykernel_12680\1337379253.py:5: SyntaxWarning: invalid escape sequence '\w'
  alphanumeric = lambda x: re.sub('\w*\d\w*', ' ', x)


,sentiment,reviews
0,negative,i wanted to love this i was even prepared for...
1,positive,grove square cappuccino cups were excellent t...
2,negative,i bought the grove square hazelnut cappuccino ...
3,negative,i love my keurig and i love most of the keuri...
4,negative,it s a powdered drink no filter in k cup br ...


### Prepare data for modeling

In [19]:

X = data.reviews
y = data.sentiment

In [21]:
# Split the data
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [29]:
# Die erste Dokument-Term-Matrix hat die Standardwerte des Count-Vectorizers – Häufigkeiten von Unigrammen.
from sklearn.feature_extraction.text import CountVectorizer

cv1 = CountVectorizer(stop_words='english')

X_train_cv1 = cv1.fit_transform(X_train)
X_test_cv1  = cv1.transform(X_test)

pd.DataFrame(X_train_cv1.toarray(), columns=cv1.get_feature_names_out()).head()

,able,abomination,absolute,absolutely,acceptable,accident,actual,actually,add,added,...,ya,year,years,yes,yessiree,yesterday,york,yuck,yum,yummy
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [33]:
#Die zweite Dokument-Term-Matrix enthält sowohl Unigramme als auch Bigramme und verwendet Indikatoren anstelle von Häufigkeiten.
cv2 = CountVectorizer(ngram_range=(1,2), binary=True, stop_words='english')

X_train_cv2 = cv2.fit_transform(X_train)
X_test_cv2  = cv2.transform(X_test)

pd.DataFrame(X_train_cv2.toarray(), columns=cv2.get_feature_names_out()).head()

,able,able cappuccino,able drink,able finish,able longer,able make,able return,able switch,abomination,abomination bet,...,yummy gas,yummy great,yummy kuerig,yummy perfect,yummy price,yummy run,yummy strong,yummy suitable,yummy treat,yummy won
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Try classifying using Logistic Regression

In [36]:
# logistic regression
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression()

In [38]:
# Das erste Modell trainieren.
lr.fit(X_train_cv1, y_train)
y_pred_cv1 = lr.predict(X_test_cv1)

In [40]:
# Das zweite Modell trainieren.
lr.fit(X_train_cv2, y_train)
y_pred_cv2 = lr.predict(X_test_cv2)

In [42]:
# Erstelle eine Funktion zur Berechnung der Fehlerkennzahlen, da wir dies mehrmals tun werden.
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

def conf_matrix(actual, predicted):
    cm = confusion_matrix(actual, predicted)
    sns.heatmap(cm, xticklabels=['predicted_negative', 'predicted_positive'], 
                yticklabels=['actual_negative', 'actual_positive'], annot=True,
                fmt='d', annot_kws={'fontsize':20}, cmap="YlGnBu");

    true_neg, false_pos = cm[0]
    false_neg, true_pos = cm[1]

    accuracy = round((true_pos + true_neg) / (true_pos + true_neg + false_pos + false_neg),3)
    precision = round((true_pos) / (true_pos + false_pos),3)
    recall = round((true_pos) / (true_pos + false_neg),3)
    f1 = round(2 * (precision * recall) / (precision + recall),3)

    cm_results = [accuracy, precision, recall, f1]
    return cm_results

In [44]:
#  Die Heatmap für das erste logistische Regressionsmodell.
cm1 = conf_matrix(y_test, y_pred_cv1)

In [46]:
# Die Heatmap für das zweite logistische Regressionsmodell.
cm2 = conf_matrix(y_test, y_pred_cv2)

In [48]:
#Fehlerkennzahlen in einem DataFrame zusammenstellen, um sie zu vergleichen.
results = pd.DataFrame(list(zip(cm1, cm2)))
results = results.set_index([['Accuracy', 'Precision', 'Recall', 'F1 Score']])
results.columns = ['LogReg1', 'LogReg2']
results

,LogReg1,LogReg2
Accuracy,0.858,0.871
Precision,0.899,0.882
Recall,0.915,0.957
F1 Score,0.907,0.918


Comparing the two models, the first model has better precision, while the second model has better accuracy and recall.

### Try classifying using Naive Bayes

In [77]:
# Das erste Naive-Bayes-Modell anpassen.
from sklearn.naive_bayes import MultinomialNB

mnb = MultinomialNB()
mnb.fit(X_train_cv1, y_train)

y_pred_cv1_nb = mnb.predict(X_test_cv1)

In [79]:
# Das zweite Naive-Bayes-Modell anpassen.
from sklearn.naive_bayes import BernoulliNB

bnb = BernoulliNB()
bnb.fit(X_train_cv2, y_train)

y_pred_cv2_nb = bnb.predict(X_test_cv2)

In [56]:
# Die Heatmap für das erste Naive-Bayes-Modell.
cm3 = conf_matrix(y_test, y_pred_cv1_nb)

In [58]:
# Die Heatmap für das zweite Naive-Bayes-Modell.
cm4 = conf_matrix(y_test, y_pred_cv2_nb)

In [60]:

results_nb = pd.DataFrame(list(zip(cm3, cm4)))
results_nb = results_nb.set_index([['Accuracy', 'Precision', 'Recall', 'F1 Score']])
results_nb.columns = ['NB1', 'NB2']
results_nb

results = pd.concat([results, results_nb], axis=1)
results

,LogReg1,LogReg2,NB1,NB2
Accuracy,0.858,0.871,0.884,0.761
Precision,0.899,0.882,0.909,0.760
Recall,0.915,0.957,0.940,1.000
F1 Score,0.907,0.918,0.924,0.864


### Try using TF-IDF instead of Count Vectorizer

In [64]:
# Erstelle TF-IDF-Versionen der zuvor im Übungsabschnitt erstellten Count-Vectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf1 = TfidfVectorizer(stop_words='english')
X_train_tfidf1 = tfidf1.fit_transform(X_train)
X_test_tfidf1  = tfidf1.transform(X_test)

tfidf2 = TfidfVectorizer(ngram_range=(1,2), binary=True, stop_words='english')
X_train_tfidf2 = tfidf2.fit_transform(X_train)
X_test_tfidf2  = tfidf2.transform(X_test)

In [66]:
# Das erste logistische Regressionsmodell auf die TF-IDF-Daten anpassen.
lr.fit(X_train_tfidf1, y_train)
y_pred_tfidf1_lr = lr.predict(X_test_tfidf1)
cm5 = conf_matrix(y_test, y_pred_tfidf1_lr)

In [68]:
# Das zweite logistische Regressionsmodell auf die TF-IDF-Daten anpassen
lr.fit(X_train_tfidf2, y_train)
y_pred_tfidf2_lr = lr.predict(X_test_tfidf2)
cm6 = conf_matrix(y_test, y_pred_tfidf2_lr)

In [70]:
# Das erste Naive-Bayes-Modell auf die TF-IDF-Daten anpassen.
mnb.fit(X_train_tfidf1.toarray(), y_train)
y_pred_tfidf1_nb = mnb.predict(X_test_tfidf1)
cm7 = conf_matrix(y_test, y_pred_tfidf1_nb)

In [72]:
# Das zweite Naive-Bayes-Modell auf die TF-IDF-Daten anpassen.
bnb.fit(X_train_tfidf2.toarray(), y_train)
y_pred_tfidf2_nb = bnb.predict(X_test_tfidf2)
cm8 = conf_matrix(y_test, y_pred_tfidf2_nb)

In [74]:

results_tf = pd.DataFrame(list(zip(cm5, cm6, cm7, cm8)))
results_tf = results_tf.set_index([['Accuracy', 'Precision', 'Recall', 'F1 Score']])
results_tf.columns = ['LR1-TFIDF', 'LR2-TFIDF', 'NB1-TFIDF', 'NB2-TFIDF']
results_tf

results = pd.concat([results, results_tf], axis=1)
results

,LogReg1,LogReg2,NB1,NB2,LR1-TFIDF,LR2-TFIDF,NB1-TFIDF,NB2-TFIDF
Accuracy,0.858,0.871,0.884,0.761,0.845,0.755,0.781,0.761
Precision,0.899,0.882,0.909,0.760,0.830,0.755,0.775,0.760
Recall,0.915,0.957,0.940,1.000,1.000,1.000,1.000,1.000
F1 Score,0.907,0.918,0.924,0.864,0.907,0.860,0.873,0.864


### Ergebnisse:

Das Multinomial Naive Bayes-Modell mit einfacher Unigram-Vektorisierung (CountVectorizer) erzielte die beste Leistung:

Accuracy: 88,4%, F1-Score: 92,4%.

### TF-IDF verbesserte den Recall, war aber insgesamt weniger präzise.

Die Verwendung von Bigrams führte zu mehr Features, aber keiner signifikanten Steigerung der Modellqualität.